# Chapter 7: Stages 1-2 of Instruction Fine-Tuning

This notebook implements chapter 7, stages 1 and 2 of the book [Build A Large Language Model From Scrath by Sebastian Raschka](https://github.com/rasbt/LLMs-from-scratch). Additionally, appendix D and E are implemented too. Chapter 7 takes the open/pretrained weights of GPT-2 and fine-tunes them on a domain specific instruction set. This is the main step behind developing LLMs for domain specific tasks such as chatbots, personal assistants, AI agents, etc.  This process is broken down into three stages:
*   **Stage 1: Preparing the dataset**
*   **Stage 2: Fine-Tuning the Pretrained LLM**
*   **Stage 3: Evaluating the LLM**




## Stage 1: Preparing the Dataset

Preparing the Dataset for fine-tuning a model using the [PyTorch](https://pytorch.org/) library consists of 1) downloading the dataset and formatting it; 2)  batching the dataset; and 3) **creating  data loaders**.




### 1. Downloading the dataset and Formatting it

#### Stanford Alpaca

The [Stanford Alpaca](https://github.com/tatsu-lab/stanford_alpaca) dataset contains 52K instruction based prompts. It consists of an **instruction, response, and sometimes an input**. After filtering out the prompts that contained inputs the dataset used for fine-tuning consisted of 31,323 examples. Of which 90% was set aside for fine-tuning, 5% for validation and 5% for later evaluating the model. Using this dataset I was able to fine-tune the large model architecture of GPT-2 consisting of 774M million parameters for 3 epochs and not overfit!!!

In [ ]:
# Helper function to download and load the Alpaca dataset
import json
import os
import requests
def download_and_load_file(file_path, url):
  if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
      file.write(text_data)
  with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)
  return data

In [ ]:
# file path to dump json file
file_path = "alpaca_data.json"
# url of github repo containing Alpaca data
url = (
    "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/refs/heads/main/alpaca_data.json"
)
# calling above function to download and load data
data_one = download_and_load_file(file_path, url)
# printing number of examples
print("Number of entries:", len(data_one))

In [ ]:
#Function to filer out prompts that contain an input
#so data only contains instructions and responses
def filter_alpacha(data_one):
  s = []
  for i in data_one:
    if i['input'] =='':
      s.append(i)
  return s
fil_data = filter_alpacha(data_one)

In [ ]:
#Printing out filtered dataset
len(fil_data)

31323

In [ ]:
# Printing out example format of Alpaca data
fil_data[0]

In [ ]:
#Printing out instruction and response found in str format
print(f"Instruction: {fil_data[0]['instruction']}")
print(f"Output: {fil_data[0]['output']}")

In [ ]:
# Spliting data into training, testing and validation
train_portion = int(len(fil_data) * 0.90)
test_portion = int(len(fil_data) * 0.05)
val_portion = len(fil_data) - train_portion - test_portion

train_data_al = fil_data[:train_portion]
test_data_al = fil_data[train_portion:train_portion + test_portion]
val_data_al = fil_data[train_portion+test_portion:]

#### LIMA

At first I tried using the [LIMA](https://huggingface.co/buckets/DexHeim/lima-bucket) dataset which was created for the paper [LIMA:Less Is More for Alignment](https://arxiv.org/pdf/2305.11206). It consists of 1,000 carefully curated prompts and responses. The paper used a **65 billion** parameter [model](https://en.wikipedia.org/wiki/Llama_(language_model)) to fine-tune the model. In short they got amazing results with such a small dataset!! (adding evidence to the [Superficial Alignment Hypothesis](https://arxiv.org/pdf/2410.03717) which is still an open question). The largest open-weight model released by [OpenAI](https://openai.com/) at the time was GPT-2-xl which has over 1 billion parameters. Intially, I used this model to fine-tune on the LIMA dataset. However, with only 1,000 instructions the model (with over 1 billion paramaters) was overfitting the data after the second epoch!!! Consequently, I decided not to use the [LIMA](https://huggingface.co/buckets/DexHeim/lima-bucket) dataset for further instruction fine-tuning. This section contains the commented out code I intially used to download, load, and preprocess LIMA for fine-tuning. I have kept it here for now, but once I refactor the code I will probably move this to a seperate notebook.  

In [ ]:
# !pip install jsonlines

In [ ]:
# Download this bucket to a local folder if it does not exist
# !hf sync hf://buckets/DexHeim/lima-bucket ./local

In [ ]:
# Downloading dataset of LIMA instructions
# import jsonlines
# import json
# file_path = "local/train.jsonl"

# def download_loadfile(file):
#   data = []
#   with open(file, 'r') as f:
#     for line in f:
#         data.append(json.loads(line.strip()))
#   return data

# data_two = download_loadfile(file_path)

In [ ]:
# len(data_two)

In [ ]:
# data_two[0]

In [ ]:
# # sample of instuctions in dataset
# print(f"Instruction: {data_two[0]['conversations'][0]}")
# print(f"Output: {data_two[0]['conversations'][1]}")

In [ ]:
# partitioning the dataset
# train_portion = int(len(data_two) * 0.95)
# test_portion = int(len(data_two) * 0.0)
# val_portion = len(data_two) - train_portion - test_portion

# train_data = data_two[:train_portion]
# test_data = data_two[train_portion:train_portion + test_portion]
# val_data = data_two[train_portion+test_portion:]

#### Alpaca Prompt Format

Below are two functions to transform the instruction datasets that were previously downloaded into the [Alpaca Prompt format](https://blog.devgenius.io/fine-tuning-my-first-llm-e7440f433c47) that is used for fine-tuning the model (i.e., converting JSON dictionaries into Alpaca prompts). I decided to use the Alpaca prompt format rather than [Microsoft's Phi-3 format](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) since I used the Alpaca dataset and the Alpaca prompt format is simpler to understand. Namely, Alpaca's prompt format uses a structured format with defined sections for **instruction, input and response**

Phi-3's prompt format has (in my opinion) a less intuitive instruction format of **<|system|>**, **<|user|>** and **<|assistant|>** where **<|system|>** defines the model's purpose and/or persona for the **<|assistant|>** which answers the **<|user|>** request. Having said that, this instruction format is well-suitied for developing AI agents and/or personal assistants due to its use of system/user/assistant prompt roles.  



In [ ]:
#Prompt format for LIMA dataset
def format_input(entry):
  instruction_text = (
      f"Below is an instruction that describes a task."
      f"Write a response that appropriately completes the request."
      f"\n\n### Instruction:\n{entry['conversations'][0]}"
  )
  return instruction_text

#Prompt format for Alpaca dataset
def format_input_alpaca(entry):
  instruction_text = (
      f"Below is an instruction that describes a task."
      f"Write a response that appropriately completes the request."
      f"\n\n### Instruction:\n{entry['instruction']}"
  )
  return instruction_text

### 2. Batching the Dataset

Unlike the batching done for classifer fine-tuning (as done in chapter 6) a custom collate function is needed and implemented for instruction fine-tuning. Generally, a collate function is responsible for taking a list of data samples and merging them into a batch for training. For instruction fine-tuning a custom collate function is needed to handle the specific formating requirements for instruction fine-tuning; namely the steps of:  
1.   Formating JSON data into Alpaca prompt style/template
2.   Tokenizing the formatted data
3.   Adding end-of-text tokens (50256) to pad data samples to the same length
4.   Creating a list of target token IDs for the model to learn
5.   Replacing certain padding tokens in the target token IDs with -100

The first two steps are done by implementing an ```InstructionDataset ``` class which formats the input using the functions above and tokenizies the formatted data.

Steps 3, 4, and 5 are handeled by the custom collate function. The custom collate function first **pads the training examples in each batch to the same length while allowing different batches to have different lengths**. Next the custom collate function creates the target token IDs that we want the model to learn during fine-tuning.The target token IDs match the input token IDs **but are shifted one position to the right** which allows the LLM to learn how to predict the next token in a given sequence. And lastly, the padding tokens of 50256, except for the first instance, are replaced with -100 in the target token IDs. By doing so, it allows us to exclude padding tokens from contributing to the training loss calculation, ensuring that only meaningful data influences model learning.





In [ ]:
'''
InstructionDataset Class that encaptulates steps 1 and 2
'''
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

#Getting tokenizer use by GPT-2
tokenizer = tiktoken.get_encoding("gpt2")

class InstructionDataset(Dataset):
  def __init__(self, data, data_al, tokenizer):
    super().__init__()
    self.data = data
    self.data_al = data_al
    self.encoded_texts = []
    # Note: This portion commented out relates to the LIMA dataset which I first
    # combined with the Alpaca dataset; I decided it was not necessary for
    # fine-tuning
    # for entry in data:
    #   instruction = format_input(entry)
    #   response_text = f"\n\n### Response:\n{entry['conversations'][1]}"
    #   full_text = instruction + response_text
    #   self.encoded_texts.append(tokenizer.encode(full_text))
    # Portion implementing steps 1 and 2
    for entry in data_al:
      instruction = format_input_alpaca(entry) #Alpaca prompt formatting
      response_text = f"\n\n### Response:\n{entry['output']}"
      full_text = instruction + response_text #combines instruction and response
      self.encoded_texts.append(tokenizer.encode(full_text)) #tokenizes prompt

  def __getitem__(self, index):
    return self.encoded_texts[index]

  def __len__(self):
    # return len(self.data) + len(self.data_al)
    return len(self.data_al)

In [ ]:
'''
Custom collate function that encaptulates steps 3, 4, and 5
'''
def custom_collate_fn(batch,
                      pad_token_id = 50256,
                      ignore_index = -100,
                      allowed_max_length = 1024,
                      device="cpu",
                      input_mask = False
                      ):
  batch_max_length = max(len(item)+1 for item in batch) #determing longest sequence in the batch
  inputs_ls, targets_ls = [], []
  #padding the input and target sequence for each batch by the longest sequence
  for item in batch:
    new_item = item.copy()
    new_item += [pad_token_id]
    padded = (new_item+[pad_token_id]*(batch_max_length - len(new_item))) #padding the sequence
    inputs = torch.tensor(padded[:-1]) #input sequence
    targets = torch.tensor(padded[1:]) #target sequence

    #converting target padded token IDs into -100 (except for first padded token ID)
    mask = targets == pad_token_id
    indices = torch.nonzero(mask).squeeze()
    if indices.numel() > 1:
      targets[indices[1:]] = ignore_index

    #making sure input and target sequences do not exceed GPT-2's context length
    if allowed_max_length is not None:
      inputs = inputs[:allowed_max_length]
      targets = targets[:allowed_max_length]

    inputs_ls.append(inputs)
    targets_ls.append(targets)

  inputs_tensor = torch.stack(inputs_ls).to(device)
  targets_tensor = torch.stack(targets_ls).to(device)

  return inputs_tensor, targets_tensor

### 3. Creating Dataloaders

These are the dataloaders that batch the data and call the ```custom_collate_fn``` (as detailed above). The ```train_loader``` and ```val_loader``` are given to the training loop and are used to input the data into the model for training.    

In [ ]:
from functools import partial

#determines if underlying hardware contains a cuda based gpu
#if so set device to gpu else cpu
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

#using partial function to create a new version of the
#customized_collate_fn with device argument prefilled
customized_collate_fn = partial(
    custom_collate_fn,
    device=device
)

In [ ]:
#use if u have multiple devices (i.e. gpu cluster)
num_workers = 0
#Batch Size
batch_size = 5
torch.manual_seed(123)

#calling InstructionDataset Class to format training data
train_dataset = InstructionDataset(train_data_al, train_data_al, tokenizer)
#creating train Dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = True,
    drop_last = True,
    num_workers = num_workers
)
#calling InstructionDataset Class to format validation data
val_dataset = InstructionDataset(val_data_al, val_data_al, tokenizer)
#creating validation Dataloader
val_loader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)
#calling InstructionDataset Class to format testing data
test_dataset = InstructionDataset(test_data_al, test_data_al, tokenizer)
#creating testing Dataloader
test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)

In [ ]:
#prints 10 items from the train_loader containing
#the batched input and targets for the model
print("Train loader:")
count=0
for inputs, targets in train_loader:
  print(inputs.shape, targets.shape)
  count+=1
  if count == 10:
    break

## Stage 2: Fine-tuning the LLM

After all of the previous data pre-processing steps we are ready now for finetuning the LLM. My stage 2 consists of five basic steps rather than three as found in the book because I added [Low Rank Adaptation (LoRA)](https://arxiv.org/pdf/2106.09685) as an additional finetuning step and a [merge and save step of the LoRA weights](https://huggingface.co/docs/peft/v0.20.0/en/package_reference/lora#merge-lora-weights-into-the-base-model). The five steps are as follows:

*   **Loading weights of a pretrained LLM**
*   **Replacing Linear Layers with LoRA Layers (i.e. Appendix E)**
*   **Instruction fine-tuning of the LLM (with Appendix D)**
*   **Viewing Model training and Validation Loss**
*   **(Optional) Merging and Saving fine-tuned weights**



### Loading weights of a Pretrained LLM

I decided to download and load the pretrained weights of the large GPT-2 model consisting of 774M parameters rather than the medium GPT-2 model of 344M parameters as the book did. I copied the [book's](https://github.com/rasbt/LLMs-from-scratch) repository of hosted GPT-2 weights and stored them in my [bucket](https://huggingface.co/buckets/DexHeim/gpt2-from-scratch-pytorch-bucket) for later use.    

In [ ]:
# importing functions containg code of previous chapters
import os
import requests
from huggingface_hub import download_bucket_files
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch07/01_main-chapter-code/previous_chapters.py"
!touch "/content/previous_chapters.py"
file_path = "/content/previous_chapters.py"
response = requests.get(url, timeout=100)
response.raise_for_status()
text_data = response.text
with open(file_path, "w", encoding="utf-8") as file:
  file.write(text_data)

In [ ]:
# importing previous chapters code containing GPT-2 base architecture and other helper functions
from previous_chapters import GPTModel, generate, token_ids_to_text, text_to_token_ids

In [ ]:
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

# different model configuations of GPT-2
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# choosing our model config to use
model_name = "gpt2-large (774M)"

# updating base config with our model config
BASE_CONFIG.update(model_configs[model_name])

'''
Different pretrained weight configurations:
gpt2-small-124M.pth
gpt2-medium-355M.pth
gpt2-large-774M.pth
gpt2-xl-1558M.pth
'''
# setting to download and load pretrained weights of GPT-2 Large
file_name = "gpt2-large-774M.pth"

# downloading pretrained weights from my bucket and saving into colab
download_bucket_files(
    bucket_id = "DexHeim/gpt2-from-scratch-pytorch-bucket",
    files = [(f"{file_name}", f'./{file_name}')]
)


In [ ]:
# loading pretrained weights into base GPT-2 large architecture
from safetensors.torch import load_file
gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(torch.load(file_name, weights_only=True))
# printing base GPT-2 large architecture
print(gpt)

### Replacing Linear Layers with LoRA Layers (i.e. Appendix E)

One of the problems I ran into when initially finetuning the model was actually getting the model to execute on the GPU. Even when using google colab's version of NVIDIA's [G4 Card](https://cloud.google.com/blog/products/compute/g4-vms-powered-by-nvidia-rtx-6000-blackwell-gpus-are-ga) with 95GB of RAM I was still getting out-of-memory (OOM) errors!!😖 I knew that I had to decrease the number of trainable parameters so I turned to Appendix E of the [book](https://github.com/rasbt/LLMs-from-scratch) detaling its implementation of [Low Rank Adaptation (LoRA)](https://arxiv.org/pdf/2106.09685).

As the book details:

> LoRA is a technique that adapts a pretrained model to better suit a specific, often smaller dataset by **adjusting only a small subset of the model’s weight parameters.** The “low-rank” aspect refers to the mathematical concept of limiting model adjustments to a smaller dimensional subspace of the total weight parameter space. This effectively captures the most influential directions of the weight parameter changes during training.

As the [article](https://arxiv.org/pdf/2106.09685) details in greater complexity, the basic gist of it is approximating the weight update matrix of gradients calculated during backpropogation i.e. $ΔW$ with a low-rank matrix decomposition of  matricies; namely B and A. As the authors of the [article](https://arxiv.org/pdf/2106.09685) detail:

> A neural network contains many dense layers which perform matrix multiplication. The weight matrices in these layers typically have full-rank. When adapting to a specific task, [Aghajanyan et al. (2020)](https://arxiv.org/abs/2012.13255) shows that the pre-trained language models have a low “instrisic dimension” [i.e., low dimensional subspace for reparameterization] and can still learn efficiently despite a random projection to a smaller subspace. Inspired by this, we hypothesize the updates to the weights also have a low “intrinsic rank” during adaptation.

The gradient step (i.e. adjusting the weights of the model) becomes: $W_{t+1}=W_{0}-∇_{AB}L(AB_{t})$ where $ W_{0}∈\mathbb{R}^{d×k}$ represents the frozen orginal pretrained weights of the model and $∇_{AB}$ with matrices $A\in\mathbb{R}^{d×r} B∈\mathbb{R}^{r×k}$ represent the trainable weights.

Instead of the book's implementation of LoRA, I relied on the tutorial [Implementing LoRA from scratch](https://sabrresearch.com/cookbooks/implementing-lora-from-scratch-pytorch) which consists of creating the class  ```LinearWithLoRA``` This class creates and stores the matrices A and B along with the scaling factor i.e., alpha/rank. Just as in the book's implementation, matrix A is intialized using a [kaiming uniform](https://docs.pytorch.org/docs/2.13/nn.init.html) distribution and matrix B is intialized with zeros. Then in in the method ```forward``` a forward pass/forward propogation is done to add the orginal output of the pretrained linear layers i.e., $W_{0}$ with the LoRA matrices i.e. AB. This modification results in the following hidden state: $h=W_{0}x+ABx$. Lasty, in the method ```delta``` the weight update matrix containing the finetuned weights multiplied by its scaling factor is returned. This method is useful if just wanting to save the LoRA weights separately or when wanting to merge the LoRA weights with the base model's weights (as done in optional step E)   

The next implementation step is replacing each linear layer of the base model with the ```LinearWithLoRA``` layer. That means replacing the linear layers found in the masked multihead attention and feed forward layers of the transfomer and the last linear output layer of the model. Instead of using the book's function to do this swapping the linear layer I again relied on the tutorial [Implementing LoRA from scratch](https://sabrresearch.com/cookbooks/implementing-lora-from-scratch-pytorch) since I was having problems with the book's implementation of this function.





In [ ]:
# importing helper functions of previous chapters of calculating a loss and implementing a simple training loop
from previous_chapters import calc_loss_loader, train_model_simple

In [ ]:
# freezing base model's pretrained weights
for param in gpt.parameters():
  param.requires_grad = False
total_params = sum(p.numel() for p in gpt.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

In [ ]:
# class that implements LinearWithLoRA Layer
import math

class LinearWithLoRA(torch.nn.Module):
  def __init__(self, base, rank, alpha):
    super().__init__()
    self.base = base # pretrained linear layers of model
    out_features, in_features = base.weight.shape # setting in_dim, out_dim
    self.A  = torch.nn.Parameter(torch.empty(rank, in_features)) # creating matrix A
    torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)) # intializing A with Kaiming Uniform
    self.B = torch.nn.Parameter(torch.zeros(out_features, rank)) # creating matrix B intialized with zeros
    self.scale = alpha/rank # scale parameter

  def forward(self, x):
    return self.base(x) + (x @ self.A.T @ self.B.T)*self.scale # modified forward pass

  def delta(self):
    return (self.B @ self.A) * self.scale # returns only finetuned weights

In [ ]:
# function that swaps orginial linear layers with LinearWithLoRA layers
def add_lora(module, r=8, alpha=16): # input: model, rank and alpha parameters
    for name, child in list(module.named_children()):
      if isinstance(child, torch.nn.Linear):
          setattr(module, name, LinearWithLoRA(child, r, alpha))
      else:
          add_lora(child, r, alpha)

add_lora(gpt)

In [ ]:
# making sure only LinearWithLoRA layers are trainable
for m in gpt.modules():
  if isinstance(m, LinearWithLoRA):
    m.A.requires_grad_(True)
    m.B.requires_grad_(True)

In [ ]:
# printing modified GPT-2 large architecture
print(gpt)

In [ ]:
# printing total number of trainable parameters of modified GPT-2 large architecture
total_params = sum(p.numel() for p in gpt.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

### Instruction fine-tuning of the LLM (with Appendix D)

Instead of using the simple training loop of chapter 7 as found in the book [Build A Large Language Model From Scrath by Sebastian Raschka](https://github.com/rasbt/LLMs-from-scratch) I decided to implment the traing loop of Appendix D since I was intially having problems with finetuning the model on the [LIMA](https://huggingface.co/buckets/DexHeim/lima-bucket) dataset; namely, after the first epoch [overfitting](https://en.wikipedia.org/wiki/Overfitting) was occuring and hence one of the reasons why I also switched to the Alpaca dataset (side note: it would be interesting to see the performance of doing another round of finetuning the model on LIMA after being finetuned on Alpaca-maybe better results❓).

This modified training loop helps in stabilizing the training/finetuning of LLMs. As the book details three additional modifications are made to the simple training loop; namely,
1.   **Learning rate warmup**
2.   **Cosine decay**
3.   **Gradient clipping**

As the book details with respect to warming up the learning rate:

> involves gradually increasing the learning rate from a very low initial value (initial_lr) to a maximum value specified by the user (peak_lr). Starting the training with smaller weight updates decreases the risk of the model encountering large, destabilizing updates during its training phase.

This is implemented by incrementing the learning rate for a period of warmup steps from the initial_lr to the peak_lr. To do so the number of global steps taken needs to be tracked and compared to the total number of training steps as determined by multiplying the number of epochs by the number of training examples. So long as the global steps are less than the number of warmup steps the learning rate is incremented within a given optimizer e.g., [Adam](https://docs.pytorch.org/docs/2.13/generated/torch.optim.Adam.html). Once it surpasses the number of warmup steps cosine decay is then applied to the learning rate.

As the book details with respect to cosine decay:

> This method modulates the learning rate throughout the training epochs, making it follow a cosine curve after the warmup stage...cosine decay reduces (or decays) the learning rate to nearly zero, mimicking the trajectory of a half-cosine cycle. The gradual learning decrease in cosine decay aims to decelerate the pace at which the model updates its weights. This is particularly important because it helps minimize the risk of overshooting the loss minima during the training process, which is essential for ensuring the stability of the training during its later phases.

This is implemented in the else condition after the warmup steps. First a progress variable is created that represents how far within the training process we are in. Then [cosine annealing](https://arxiv.org/pdf/1608.03983) is applied to this varaible to modify the learning rate.

As the book details with respect to Gradient clipping:

> This method involves setting a threshold above which gradients are downscaled to a predetermined maximum magnitude. This process ensures that the updates to the model’s parameters during backpropagation stay within a manageable range.

This is implemented by applying the [clip gradient norm function](https://docs.pytorch.org/docs/2.13/generated/torch.nn.utils.clip_grad_norm_.html) after calculating the gradients i.e. ```loss.backward()``` but before taking the gradient descent step i.e. ```optimizer.step()```



In [ ]:
# function that implements the modified training loop
from previous_chapters import evaluate_model, generate_and_print_sample, calc_loss_batch
import time
def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):
  train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], [] # tracking losses, tokens seen, and learning rates
  tokens_seen, global_step = 0, -1 # setting intial values of tokens seen and global step

  peak_lr = optimizer.param_groups[0]["lr"] # getting the peak learning rate from the optimizer
  total_training_steps = len(train_loader) * n_epochs # calc total training steps
  lr_increment = (peak_lr-initial_lr)/warmup_steps # calc learning rate increment

  for epoch in range(n_epochs):
    model.train()
    for input_batch, target_batch in train_loader:
      optimizer.zero_grad() # zeroing the gradients
      global_step += 1 # incrementing global step
      if global_step < warmup_steps:     # warmup period
        lr = initial_lr + global_step * lr_increment
      else:                              # cosine decay
        progress = ((global_step-warmup_steps)/
                    (total_training_steps - warmup_steps))
        lr = min_lr + (peak_lr - min_lr) * 0.5 * (
           1 + math.cos(math.pi * progress))

      for param_group in optimizer.param_groups:   # adjusting optimizer's learning rate
        param_group['lr'] = lr
      track_lrs.append(lr)
      loss = calc_loss_batch(input_batch, target_batch, model, device) # calc loss
      loss.backward()                               # calc gradients

      if global_step >= warmup_steps:               # clipping the gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

      optimizer.step()                              # gradient step
      tokens_seen += input_batch.numel()            # num of tokens seen so far

      if global_step % eval_freq == 0:              # validation portion
        train_loss, val_loss = evaluate_model(
            model, train_loader, val_loader,
            device, eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
        )

    generate_and_print_sample(                       # printing generated response
        model, tokenizer, device, start_context
    )
  return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
# determine if gpu is available
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
# moving GPT-2 model to device
gpt.to(device)
# start clock
start_time = time.time()
# reproducibility
torch.manual_seed(123)
# optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), weight_decay=0.1)
# epochs
num_epochs = 1
# calc warmup period
warmup_steps = int(0.2 * (len(train_loader) * num_epochs)) # 20% warmup
# calling modified training loop
train_losses, val_losses, tokens_seen, lrs = train_model(
    gpt, train_loader, val_loader, optimizer, device,
    n_epochs=num_epochs, eval_freq=1000, eval_iter=2,
    start_context=format_input_alpaca(val_data_al[0]), tokenizer=tokenizer,
    warmup_steps=warmup_steps, initial_lr=1e-4, min_lr=1e-5
)
# end clock
end_time = time.time()
execution_time_min = (end_time - start_time) / 60
print(f"Training completed in {execution_time_min:.2f} minutes.")

In [ ]:
# 4. Viewing Model training and Validation Loss
from previous_chapters import plot_losses
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

### (Optional) Merging LoRA Weights and Saving Fine-tuned Model

Because LoRA changes the base configuration of the gpt-2 model (i.e., changes the Linear layers to LinearWithLoRA layers) another step has to be done before the model's state and weights can be saved for later use. The orginal gpt-2 model was not trained using the LoRA method and hence it will output a matching error if trying to load the LoRA based weights into the orginal base configuation. One way to overcome this problem is to merge the LoRA weights back into the orginal frozen weights of the model and then remove the LinearWithLoRA layers. Generally, when using [HuggingFace's LoRA implementation](https://huggingface.co/docs/peft/package_reference/lora) this step is done by the function ```merge_and_unload()```  which merges the trained LoRA adapter weights directly into the base model and unloads the adapter layers. At first I tried to duplicate this function, but after looking at the [source code](https://github.com/huggingface/peft/blob/v0.20.0/src/peft/tuners/tuners_utils.py#L696) I was lost 😵. Luckily I stumbled across [SABR Research's](https://sabrresearch.com/) cookbook of [Implementing LoRA from Scratch on a Toy Model](https://sabrresearch.com/cookbooks/implementing-lora-from-scratch-pytorch) which contains a section regarding merging LoRA weights. As detailed by the section [Merge the adapter](https://sabrresearch.com/cookbooks/implementing-lora-from-scratch-pytorch):

> When fine-tuning is done, you can fold the update back into the frozen weights of the base model. Since the scaled matrix of B·A has the same shape as the orginal weights of the base model W, W + ΔW is an ordinary weight, and the wrapped layer produces the same outputs with the adapter zeroed. Merging removes the inference overhead and lets you ship a plain model...[t]he merged model is very close to the original model.

After merging the LoRA weights, all that needs to be done next is to replace the LinearWithLoRA layers with the model's orginal Linear layers. I did this by sub-classing Pytorch's [Linear](https://docs.pytorch.org/docs/2.13/generated/torch.nn.Linear.html) layer so that it stores the fine-tuned weights and replaced each LinearWithLoRA layer with this modified Linear layer. After doing so, the model's state and weights are saved and uploaded to my [Bucket](https://huggingface.co/buckets/DexHeim/gpt2-large-774M-ALPACA-sft) for later use (i.e. evaluation of the fine-tuned model).







In [ ]:
# subclassing torch.nn.Linear function
import torch.nn as nn
class Linear(nn.Linear):
  def __init__(self, in_features, out_features, weight, bias=True):
    super().__init__(in_features, out_features, bias=bias)
    # storing the finetuned weights
    self.weight = weight

  def forward(self, x):
    base_output = super().forward(x) # forward pass
    return base_output

In [ ]:
# function to merge AB weights with base model
@torch.no_grad()
def merge():
  for m in gpt.modules():
    if isinstance(m, LinearWithLoRA):
      m.base.weight += m.delta()
      m.B.zero_()
merge()

In [ ]:
# swapping LinearWithLoRA layers with Linear layers
# most of code snippet taken from https://lightning.ai/lightning-ai/templates/code-lora-from-scratch?section=featured
# but was done for implementing LoRA not removing LoRA layers/adapters from base model

from functools import partial

# default hyperparameter choices
lora_query = True
lora_key = True
lora_value = True
lora_projection = True
lora_mlp = True
lora_head = True

layers = []

# assigning Linear class with args
assign_att = partial(Linear, in_features=1280, out_features=1280, bias=True)
assign_ff_one =  partial(Linear, in_features=1280, out_features=5120, bias=True)
assign_ff_two =  partial(Linear, in_features=5120, out_features=1280, bias=True)
assign_out_head = partial(Linear, in_features=1280, out_features=50257, bias=False)

# replacing each LinearWithLoRA layer of the transformer with assigned Linear class with args
for layer in gpt.trf_blocks:
    if lora_query:
        layer.att.W_query = assign_att(weight=layer.att.W_query.base.weight)
    if lora_key:
        layer.att.W_key = assign_att(weight=layer.att.W_key.base.weight)
    if lora_value:
        layer.att.W_value = assign_att(weight=layer.att.W_value.base.weight)
    if lora_projection:
        layer.att.out_proj = assign_att(weight=layer.att.out_proj.base.weight)
    if lora_mlp:
        layer.ff.layers[0] = assign_ff_one(weight=layer.ff.layers[0].base.weight)
        layer.ff.layers[2] = assign_ff_two(weight=layer.ff.layers[2].base.weight)
# replacing LinearWithLoRA layer of the model with assigned Linear class with args
if lora_head:
  gpt.out_head = assign_out_head(weight=gpt.out_head.base.weight)

In [ ]:
print(gpt)

In [ ]:
# saving finetuned model state
torch.save(gpt.state_dict(), "/content/drive/MyDrive/GPT-2/GPT-2-755M-LoRA-Alpaca-weights.pth")

In [ ]:
# upload saved finetuned model to my bucket
!hf auth login
!hf sync /content/drive/MyDrive/GPT-2/ hf://buckets/DexHeim/gpt2-large-774M-ALPACA-sft